# LTX Hindi Avatar Image-to-Video

Use this notebook in Google Colab to create a 5-15 second image-to-video clip from one avatar image, add Hindi voice audio, preview it, and download the result.

Use fictional or consenting adult avatars only. Do not impersonate real people deceptively or create non-consensual sexual content.

## 1. Check GPU

Runtime -> Change runtime type -> GPU, then run this cell.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Install libraries

This can take a few minutes. Run once per Colab session.

In [ ]:
!pip -q install -U git+https://github.com/huggingface/diffusers.git
!pip -q install -U transformers accelerate safetensors imageio imageio-ffmpeg pillow edge-tts

## 3. Upload avatar image

Upload a clear front-facing or slight 3/4 avatar image. Talking videos work best when the face is visible and not turned far sideways.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
INPUT_IMAGE = next(iter(uploaded.keys()))
print('Using image:', INPUT_IMAGE)

## 4. Settings

Start with 5 seconds. After it works, try 10 or 15 seconds. Vertical is best for Reels/Shorts.

In [ ]:
VIDEO_STYLE = 'vertical_9_16' #@param ['vertical_9_16', 'wide_16_9', 'square']
SECONDS = 5 #@param {type:'slider', min:5, max:15, step:1}
FPS = 24 #@param {type:'slider', min:16, max:24, step:1}
SEED = 12345 #@param {type:'integer'}

if VIDEO_STYLE == 'vertical_9_16':
    WIDTH, HEIGHT = 480, 704
elif VIDEO_STYLE == 'wide_16_9':
    WIDTH, HEIGHT = 704, 480
else:
    WIDTH, HEIGHT = 576, 576

frames = int(SECONDS * FPS)
NUM_FRAMES = ((frames - 1) // 8) * 8 + 1
print('Resolution:', WIDTH, 'x', HEIGHT)
print('Frames:', NUM_FRAMES, 'Duration:', round(NUM_FRAMES / FPS, 2), 'seconds')

## 5. Prompt and Hindi speech

Keep the video prompt in English for better control. Put the spoken dialogue in Hindi.

In [ ]:
PROMPT = '''A realistic close-up portrait video of an adult Indian woman avatar speaking warmly to the camera. She keeps natural eye contact, blinks softly, makes subtle head movements, and smiles gently. The camera is stable, portrait composition, shallow depth of field, clean studio lighting, realistic skin texture, natural facial motion, no exaggerated expression.'''

NEGATIVE_PROMPT = '''worst quality, low quality, blurry, distorted face, deformed mouth, extra teeth, bad eyes, crossed eyes, jitter, flicker, warped face, unnatural skin, plastic skin, duplicate face, broken hands, text, watermark'''

HINDI_TEXT = '''नमस्ते, मैं आपकी डिजिटल अवतार हूं। आज मैं आपसे हिंदी में बात कर रही हूं। यह वीडियो एक इमेज और प्रॉम्प्ट से बनाया गया है।'''

print(PROMPT)
print('\nHindi speech:')
print(HINDI_TEXT)

## 6. Generate image-to-video with LTX

This uses the practical Colab-friendly LTX distilled model. On paid A100/H100 you can experiment later with LTX-2.3, but this path is the most likely to run successfully anywhere.

In [ ]:
import torch
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXConditionPipeline, LTXVideoCondition
from diffusers.utils import load_image, export_to_video

assert torch.cuda.is_available(), 'Please enable GPU in Colab runtime.'
major, minor = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if major >= 8 else torch.float16
print('Using dtype:', dtype)

model_id = 'Lightricks/LTX-Video-0.9.7-distilled'
pipe = LTXConditionPipeline.from_pretrained(model_id, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

image = load_image(INPUT_IMAGE).convert('RGB')
condition = LTXVideoCondition(image=image, frame_index=0)
generator = torch.Generator(device='cuda').manual_seed(SEED)

video = pipe(
    conditions=[condition],
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_frames=NUM_FRAMES,
    timesteps=[1000, 993, 987, 981, 975, 909, 725, 0.03],
    decode_timestep=0.05,
    decode_noise_scale=0.025,
    image_cond_noise_scale=0.025,
    guidance_scale=1.0,
    guidance_rescale=0.7,
    generator=generator,
    output_type='pil',
).frames[0]

export_to_video(video, 'ltx_avatar_raw.mp4', fps=FPS)
print('Saved: ltx_avatar_raw.mp4')

In [ ]:
from IPython.display import Video, display
display(Video('ltx_avatar_raw.mp4', embed=True, width=360))

## 7. Generate Hindi voice

In [ ]:
import edge_tts

VOICE = 'hi-IN-SwaraNeural' # female Hindi voice
communicate = edge_tts.Communicate(HINDI_TEXT, VOICE)
await communicate.save('hindi_voice.mp3')
print('Saved: hindi_voice.mp3')

## 8. Merge audio and video

This adds Hindi audio. For stronger mouth sync, use the optional LatentSync section below on a strong GPU.

In [ ]:
!ffmpeg -y -i ltx_avatar_raw.mp4 -i hindi_voice.mp3 -map 0:v:0 -map 1:a:0 -shortest -c:v libx264 -pix_fmt yuv420p -c:a aac -b:a 192k avatar_with_hindi_audio.mp4

from IPython.display import Video, display
display(Video('avatar_with_hindi_audio.mp4', embed=True, width=360))

## 9. Download result

In [ ]:
from google.colab import files
files.download('avatar_with_hindi_audio.mp4')

## Optional: LatentSync lip sync

Run this only if you have enough GPU VRAM. LatentSync 1.5 needs about 8 GB; LatentSync 1.6 is sharper but needs about 18 GB. If setup fails on free Colab, keep the audio-merged result above or switch to A100.

In [ ]:
# Optional heavy section. Uncomment and run on a strong GPU.
# %cd /content
# !git clone https://github.com/bytedance/LatentSync.git
# %cd /content/LatentSync
# !bash setup_env.sh
# !python -m scripts.inference \
#   --unet_config_path 'configs/unet/stage2_512.yaml' \
#   --inference_ckpt_path 'checkpoints/latentsync_unet.pt' \
#   --inference_steps 20 \
#   --guidance_scale 1.5 \
#   --enable_deepcache \
#   --video_path '/content/ltx_avatar_raw.mp4' \
#   --audio_path '/content/hindi_voice.mp3' \
#   --video_out_path '/content/final_hindi_lipsync.mp4'
# from IPython.display import Video, display
# display(Video('/content/final_hindi_lipsync.mp4', embed=True, width=360))
# from google.colab import files
# files.download('/content/final_hindi_lipsync.mp4')